# FEATURE ENGINEERING

In [12]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import ast

This notebook performs the feature engineering process used throughout the thesis. It consists of four main stages:

1. Loading and merging the cleaned datasets.
2. Creating the prediction target.
3. Engineering predictive features.
4. Visualizing the engineered features.

The feature engineering part is based on both the EDA and the literature review.

### LOAD THE DATA AND SET INDEX

We begin by loading all cleaned datasets and standardizing their indices. 
For each dataset, we inspect its temporal coverage, number of countries, number of observations, duplicated records, and possible temporal gaps. These checks ensure that all sources are internally consistent before they are merged.

In [13]:
# ACLED
acled = pd.read_csv("../data_clean/acled_clean.csv")
acled = acled.set_index(['iso3', 'month']).sort_index()

# IDMC
idmc = pd.read_csv("../data_clean/idmc_clean.csv")
idmc = idmc.set_index(['iso3', 'month']).sort_index()

# HDX
hdx = pd.read_csv("../data_clean/hdx_clean.csv")
hdx = hdx.set_index(['iso3', 'month']).sort_index()

# ECONAI
econAI = pd.read_csv("../data_clean/econAI_clean.csv")
econAI = econAI.set_index(['iso3', 'month']).sort_index()

# INFORM INDEX
inform = pd.read_csv("../data_clean/inform_clean.csv")
inform = inform.set_index(['iso3', 'month']).sort_index()

In [14]:
datasets = {
    "IDMC": idmc,
    "ACLED": acled,
    "HDX": hdx,
    "EconAI": econAI,
    "INFORM Index": inform
}

def analyze_datasets(datasets_dict):
    analysis = []
    
    for name, df in datasets_dict.items():
       
        temp_df = df.copy()
        temp_df = temp_df.reset_index()
        temp_df['month'] = pd.to_datetime(temp_df['month'])
        
        analysis.append({
            "Dataset": name,
            "Rows": len(temp_df),
            "Countries": temp_df['iso3'].nunique(),
            "Min_Date": temp_df['month'].min().strftime('%Y-%m'),
            "Max_Date": temp_df['month'].max().strftime('%Y-%m'),
            "Avg_Months": round(len(temp_df) / temp_df['iso3'].nunique(), 1)
        })
    
    return pd.DataFrame(analysis)

df_info = analyze_datasets(datasets)
print(df_info.to_string(index=False))

     Dataset  Rows  Countries Min_Date Max_Date  Avg_Months
        IDMC  8844         89  2018-01  2026-04        99.4
       ACLED 25236        238  1996-12  2026-03       106.0
         HDX  1209        107  1998-05  2026-03        11.3
      EconAI 35472        182  2010-01  2026-03       194.9
INFORM Index 25212        191  2016-05  2027-04       132.0


In [15]:
def check_temporal_gaps(df):
    df = df.copy().reset_index()
    df['month'] = pd.to_datetime(df['month'])
    df = df.sort_values(['iso3', 'month'])
    
    df['diff'] = df.groupby('iso3')['month'].diff() / pd.Timedelta(days=31)
    
    gaps = df[df['diff'] > 1.1]
    
    if gaps.empty:
        print("No temporal gaps found in the dataset.")
    else:
        print(f"{len(gaps)} temporal gaps found:")
        print(gaps[['iso3', 'month', 'diff']].head(10))
    return gaps

gaps_df = check_temporal_gaps(idmc)
gaps_df = check_temporal_gaps(acled)
gaps_df = check_temporal_gaps(hdx)
gaps_df = check_temporal_gaps(econAI)
gaps_df = check_temporal_gaps(inform)

No temporal gaps found in the dataset.
2512 temporal gaps found:
   iso3      month      diff
1   ABW 2018-09-01  5.935484
2   ABW 2019-02-01  4.935484
4   ABW 2019-06-01  2.967742
8   ABW 2020-03-01  5.870968
9   ABW 2020-05-01  1.967742
11  ABW 2020-08-01  1.967742
12  ABW 2020-10-01  1.967742
14  ABW 2021-01-01  1.967742
15  ABW 2021-03-01  1.903226
16  ABW 2021-09-01  5.935484
965 temporal gaps found:
   iso3      month       diff
2   AFG 2018-02-01  10.870968
3   AFG 2018-05-01   2.870968
5   AFG 2018-09-01   2.967742
6   AFG 2019-03-01   5.838710
7   AFG 2019-09-01   5.935484
8   AFG 2020-04-01   6.870968
9   AFG 2020-08-01   3.935484
11  AFG 2021-04-01   6.838710
14  AFG 2021-10-01   3.935484
16  AFG 2022-06-01   6.838710
No temporal gaps found in the dataset.
No temporal gaps found in the dataset.


Temporal gaps are expected for ACLED and HDX Signals because these datasets only record months in which an event or an alert occurred. Therefore, missing months correspond to periods with no recorded events rather than missing observations.

Regarning the missing data:

- ACLED: For countries included in the analysis, missing ACLED observations indicate that no conflict events were recorded during that month, so we can impute them easily. We can also impute gaps, because months with missing values are months without events.
- IDMC: For countries included in the analysis, missing IDMC observations indicate that no displacements were recorded during that month, so we can impute them easily. We can't impute a gap but yes a whole country.
- HDX Signals: Missing values in the period covered by HDX means that nothing happened there, so can be imputed (even a whole country).
- ECONAI: We can't impute anything.
- INFORM Index: We can't impute anything

Based on that our dataset should contain only the countries contained in the intersection on ECONAI and INFORM index. Regarding the time periods, we need to keep only the periods covered by all the datasets, that is >=2018.

In [16]:
# Get the strict intersection of countries present in BOTH EconAI and INFORM
econAI_countries = set(econAI.reset_index()['iso3'].unique())
inform_countries = set(inform.index.get_level_values('iso3').unique())

# The target list: countries that MUST be in both datasets
target_countries = econAI_countries.intersection(inform_countries)

print("=" * 70)
print(f"TARGET COUNTRIES (Intersection of EconAI & INFORM): {len(target_countries)}")
print("=" * 70)

# Check how many countries from each dataset are discarded/lost based on this target
for name, df_actual in datasets.items():
    actual_countries = set(df_actual.reset_index()['iso3'].unique())
    
    # Countries that are in the current dataset but WILL BE LOST 
    # because they are not in our target intersection
    discarded = actual_countries - target_countries
    
    # Countries that this dataset lacks to reach the target (if any)
    missing_from_target = target_countries - actual_countries
    
    print(f"{name}:")
    print(f"   • Total countries in raw file: {len(actual_countries)}")
    print(f"   • Kept for analysis: {len(actual_countries.intersection(target_countries))}")
    
    if len(discarded) > 0:
        print(f"   Discarded countries (not in intersection): {len(discarded)} {sorted(list(discarded))}")
    else:
        print("   Perfect! No countries discarded from this dataset.")
        
    if len(missing_from_target) > 0:
        print(f"   Lacks these target countries (will cause NaNs): {sorted(list(missing_from_target))}")
        
    print("-" * 70)

TARGET COUNTRIES (Intersection of EconAI & INFORM): 176
IDMC:
   • Total countries in raw file: 89
   • Kept for analysis: 86
   Discarded countries (not in intersection): 3 ['AB9', 'MYT', 'NCL']
   Lacks these target countries (will cause NaNs): ['ALB', 'ARE', 'ARG', 'AUT', 'BEL', 'BGR', 'BHS', 'BLZ', 'BRB', 'BRN', 'BTN', 'BWA', 'CAN', 'CHE', 'CHL', 'CHN', 'CRI', 'CUB', 'CZE', 'DEU', 'DNK', 'DOM', 'DZA', 'ERI', 'ESP', 'EST', 'FIN', 'FJI', 'GAB', 'GEO', 'GNB', 'GNQ', 'GRD', 'GTM', 'GUY', 'HRV', 'HUN', 'IRL', 'ISL', 'JAM', 'JOR', 'JPN', 'KOR', 'KWT', 'LAO', 'LSO', 'LTU', 'LUX', 'LVA', 'MAR', 'MDA', 'MDV', 'MKD', 'MLT', 'MNE', 'MNG', 'MRT', 'MUS', 'MYS', 'NAM', 'NOR', 'NPL', 'NZL', 'OMN', 'PAN', 'POL', 'PRK', 'PRT', 'PRY', 'RWA', 'SAU', 'SEN', 'SGP', 'SRB', 'STP', 'SVK', 'SVN', 'SWE', 'SWZ', 'SYC', 'TKM', 'TLS', 'TON', 'TTO', 'TUN', 'URY', 'UZB', 'VNM', 'VUT', 'WSM']
----------------------------------------------------------------------
ACLED:
   • Total countries in raw file: 238
   • K

In [17]:
print(target_countries)

{'SEN', 'KHM', 'TCD', 'AUT', 'SYC', 'KOR', 'HND', 'ESP', 'NER', 'MDA', 'MRT', 'LBN', 'NZL', 'EGY', 'FJI', 'VUT', 'MYS', 'BFA', 'SYR', 'SOM', 'FIN', 'ALB', 'BTN', 'PSE', 'LUX', 'COD', 'ETH', 'BDI', 'JAM', 'KGZ', 'ARG', 'GUY', 'PNG', 'BGD', 'GEO', 'DEU', 'MLI', 'PAN', 'PRT', 'JOR', 'HUN', 'ZMB', 'UKR', 'SWE', 'ARM', 'ZAF', 'MWI', 'MUS', 'KEN', 'LSO', 'GRC', 'PER', 'TJK', 'CAF', 'JPN', 'ITA', 'GAB', 'ZWE', 'LVA', 'WSM', 'DNK', 'BGR', 'UGA', 'AUS', 'GTM', 'BLR', 'LBR', 'BWA', 'MNG', 'BRN', 'POL', 'MMR', 'SLE', 'MLT', 'KAZ', 'CIV', 'NPL', 'GNQ', 'SLB', 'USA', 'LTU', 'TUR', 'AGO', 'MDG', 'KWT', 'GMB', 'NIC', 'DOM', 'CZE', 'PAK', 'UZB', 'TTO', 'EST', 'BEN', 'SVK', 'COL', 'TKM', 'BHS', 'MKD', 'CUB', 'CMR', 'HTI', 'THA', 'SVN', 'GRD', 'SDN', 'BOL', 'BEL', 'BRB', 'CHL', 'PRK', 'RUS', 'ISR', 'NLD', 'ISL', 'CHE', 'PHL', 'RWA', 'ARE', 'VNM', 'SLV', 'IRN', 'BRA', 'IND', 'NOR', 'SGP', 'CAN', 'SRB', 'AFG', 'GHA', 'CHN', 'BLZ', 'NGA', 'ECU', 'SAU', 'VEN', 'MOZ', 'AZE', 'LBY', 'MNE', 'YEM', 'TGO', 'BHR'

The final dataset is restricted to the intersection of countries available in both EconAI and INFORM, since these datasets provide variables required for all observations. The temporal coverage begins in January 2018, which corresponds to the common analysis period.

Before merging the datasets, country codes and monthly timestamps are standardized to ensure that observations are perfectly aligned across all data sources.

In [18]:
# Function to standardize the DataFrame indices and ensure consistency across datasets
def standardize_df(df):
    df = df.reset_index() 
    
    if 'iso3' in df.columns:
        df['iso3'] = df['iso3'].astype(str).str.strip().str.upper()
    
    if 'month' in df.columns:
        df['month'] = pd.to_datetime(df['month'])
        df['month'] = df['month'].dt.to_period('M').dt.to_timestamp()
        
    return df.set_index(['iso3', 'month'])

# Apply the standardization function to all datasets
econAI = standardize_df(econAI)
inform = standardize_df(inform)
acled = standardize_df(acled)
hdx = standardize_df(hdx)
idmc = standardize_df(idmc)

# Create the skeleton index for the final DataFrame
max_month_idmc = idmc.index.get_level_values('month').max()
time_range = pd.date_range(start='2018-01-01', end=max_month_idmc, freq='MS') 

skeleton_index = pd.MultiIndex.from_product(
    [sorted(list(target_countries)), time_range], 
    names=['iso3', 'month']
)

df_base = pd.DataFrame(index=skeleton_index)

# Merge all datasets into the base DataFrame
df = df_base.join(econAI, how="left") \
                  .join(inform, how="left") \
                  .join(acled, how="left") \
                  .join(hdx, how="left") \
                  .join(idmc, how="left") 

print(df.head())

                   risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.999608  1.000000       8.331437        9.280007     7.8   
     2018-02-01  0.996514  1.000000       8.295601        9.393767     7.8   
     2018-03-01  0.998218  1.000000       8.339758        9.346226     7.8   
     2018-04-01  0.995082  0.999815       8.383570        9.414723     7.8   
     2018-05-01  0.991237  0.998947       8.342653        9.481498     7.7   

                  VU   CC   HA  \
iso3 month                       
AFG  2018-01-01  7.1  7.7  8.8   
     2018-02-01  7.1  7.7  8.8   
     2018-03-01  7.1  7.7  8.8   
     2018-04-01  7.1  7.7  8.8   
     2018-05-01  7.1  7.5  8.7   

                                                        event_type  \
iso3 month                                                           
AFG  2018-01-01  ['Battles', 'Explosions/Remote violence', 'Str...   
    

A complete country-month skeleton is then created containing every country in the analysis for every month in the study period. Each dataset is subsequently left-joined onto this skeleton so that missing observations remain explicit and can be handled consistently.

In [19]:
df.size, df.shape

(281600, (17600, 16))

In [20]:
print(f"Min month: {df.index.get_level_values('month').min()}")
print(f"Max month: {df.index.get_level_values('month').max()}")
print(f"Number of countries: {df.index.get_level_values('iso3').nunique()}")

Min month: 2018-01-01 00:00:00
Max month: 2026-04-01 00:00:00
Number of countries: 176


In [21]:
nans_por_columna = df.isna().sum()
print(nans_por_columna)

risk_3                    176
risk_12                   176
logfat_risk_3             176
logfat_risk_12            176
INFORM                      0
VU                          0
CC                          0
HA                          0
event_type               3042
sub_event_type           3042
disorder_type            3042
events                   3042
fatalities               3042
hdx_alert_level         16651
hdx_value               16651
monthly_displacement     9056
dtype: int64


Let's now solve the problem of the missing values:

- Missing values in `notes_acled`, `fatalities`, `event_count`, `Battles`, `Explosions/Remote violence`, `Protests`, `Riots`, `Strategic developments` and `Violence against civilians` correspond to months x countries with no events nor fatalities, so we will put 0 on all of them, and an empty string in notes_acled. 

- Missing values in `monthly_displacement` correspond to countries that hasn't had any displacement in all the covered period, so we are putting a 0.

- Missing values in `hdx_alert_Medium concern` and `hdx_alert_High concern` correspond to the month x country with no hdx alert, so we put 0 in both. Te same for `hdx_value`, so we are putting a 0.

Missing values (NaN) in the risk indicators are not imputed because they correspond to the final month of the dataset. These observations will later be removed when constructing the prediction target.

**NOTE:** Also note that some datsets doesn't have data of 2026-04, and we are imputing it with the same logic as the gaps in the middle of the data set. This is conceptually incorrect, because it's not that nothing happended, it's just that we don't have the information. However, this is not a problem in this case, Since these final observations are excluded during model training, this temporary imputation does not affect the predictive models.

In [ ]:
df["notes_acled"] = df["notes_acled"].fillna("")
df["fatalities"] = df["fatalities"].fillna(0)
df["event_count"] = df["event_count"].fillna(0)
df["Battles"] = df["Battles"].fillna(0)
df["Explosions/Remote violence"] = df["Explosions/Remote violence"].fillna(0)
df["Protests"] = df["Protests"].fillna(0)
df["Riots"] = df["Riots"].fillna(0)
df["Strategic developments"] = df["Strategic developments"].fillna(0)
df["Violence against civilians"] = df["Violence against civilians"].fillna(0)
df["hdx_alert_Medium concern"] = df["hdx_alert_Medium concern"].fillna(0)
df["hdx_alert_High concern"] = df["hdx_alert_High concern"].fillna(0)
df["hdx_value"] = df["hdx_value"].fillna(0)
df["monthly_displacement"] = df["monthly_displacement"].fillna(0)

KeyError: 'event_count'

In [ ]:
df = df.sort_index()
df.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.494067  0.695173       7.690273        8.159422     7.8   
     2018-02-01  0.506174  0.726673       7.775459        8.036293     7.8   
     2018-03-01  0.493192  0.713110       7.890279        8.250671     7.8   
     2018-04-01  0.469532  0.713219       8.010917        8.123474     7.8   
     2018-05-01  0.483328  0.696213       8.044887        8.216409     7.7   

                  VU   CC   HA  fatalities  event_count  ... Battles  \
iso3 month                                               ...           
AFG  2018-01-01  7.1  7.7  8.8      2822.0       1145.0  ...   777.0   
     2018-02-01  7.1  7.7  8.8      1829.0        920.0  ...   555.0   
     2018-03-01  7.1  7.7  8.8      2286.0        976.0  ...   600.0   
     2018-04-01  7.1  7.7  8.8      2794.0       1245.0  ...   825.0   
     2018-05-01  7.1  7.5  8.7      4264.0       1482.0  ...   975.0   

                 Explosions/Remote violence  Protests  Riots  \
iso3 month                                                     
AFG  2018-01-01                       282.0       9.0    2.0   
     2018-02-01                       299.0      16.0    1.0   
     2018-03-01                       320.0      25.0    0.0   
     2018-04-01                       346.0      29.0    1.0   
     2018-05-01                       444.0      10.0    1.0   

                 Strategic developments  Violence against civilians  \
iso3 month                                                            
AFG  2018-01-01                    37.0                        38.0   
     2018-02-01                    23.0                        26.0   
     2018-03-01                    14.0                        17.0   
     2018-04-01                    22.0                        22.0   
     2018-05-01                    18.0                        34.0   

                 hdx_value  hdx_alert_High concern  hdx_alert_Medium concern  \
iso3 month                                                                     
AFG  2018-01-01        0.0                     0.0                       0.0   
     2018-02-01        1.0                     0.0                       1.0   
     2018-03-01        0.0                     0.0                       0.0   
     2018-04-01        0.0                     0.0                       0.0   
     2018-05-01        2.0                     1.0                       0.0   

                 monthly_displacement  
iso3 month                             
AFG  2018-01-01           7456.583577  
     2018-02-01          13918.956010  
     2018-03-01          15410.272726  
     2018-04-01          17198.881440  
     2018-05-01          39769.719735  

[5 rows x 21 columns]

### CREATE THE TARGET VARIABLE

Our objective is to predict whether a country will become eligible for a new CERF Rapid Response allocation within the following two months. To construct this prediction target, we first define an intermediate variable indicating whether a country is allocation-eligible in a given month.

A country is considered allocation-eligible if it satisfies the CERF eligibility criteria for a displacement-related Rapid Response allocation: at least 50,000 new internal displacements accumulated over a three-month period and some new displacement occurring in the current month. The second condition ensures that eligibility corresponds to an ongoing displacement situation rather than to the residual effect of previous months. These criteria are based on the CERF allocation guidelines.

Using this intermediate variable, we create the prediction target as an incidence variable that equals 1 if a country becomes allocation-eligible for the first time at any point during the following two months, and 0 otherwise. In other words, the target identifies the beginning of a new eligibility episode rather than all months in which a country remains eligible.

Finally, observations corresponding to months in which a country is already allocation-eligible, but that are not one or two months before the start of a new eligibility episode, are assigned a value of `NaN`. These observations are removed before model training instead of being treated as negative examples. This ensures that the model learns to anticipate the onset of humanitarian crises rather than recognizing situations that are already underway.

In [ ]:
df = df.reset_index()
df = df.sort_values(by=['iso3', 'month'])


df['rolling_3m_displacements'] = (
    df.groupby('iso3')['monthly_displacement']
    .rolling(window=3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

df['allocation-eligible'] = (
    (df['rolling_3m_displacements'] >= 50000) & 
    (df['monthly_displacement'] > 0)
).astype(int)

df = df.set_index(['iso3', 'month']).sort_index()

Now we can create our target variable:

In [ ]:
df["target_2m"] = (
    (df.groupby(level="iso3")["allocation-eligible"].shift(-1) == 1) |
    (df.groupby(level="iso3")["allocation-eligible"].shift(-2) == 1)
).astype(int)

df["start_conflict"] = (
    (df["allocation-eligible"] == 1) & 
    (df.groupby(level="iso3")["allocation-eligible"].shift(1).fillna(0) == 0)
).astype(int)

alert_start = (
    (df.groupby(level="iso3")["start_conflict"].shift(-1) == 1) |
    (df.groupby(level="iso3")["start_conflict"].shift(-2) == 1)
)

df.loc[(df["target_2m"] == 1) & (~alert_start), "target_2m"] = np.nan

print("Total positive targets (conflict in next 2 months):", df["target_2m"].sum())

Total positive targets (conflict in next 2 months): 212.0


In [ ]:
df["target_2m"].value_counts(dropna=False)

target_2m
0.0    16868
NaN      520
1.0      212
Name: count, dtype: int64

In [ ]:
df.to_parquet("../data_clean/complete_dataset.parquet")

### FEATURE ENGINEERING

In [ ]:
df = pd.read_parquet("../data_clean/complete_dataset.parquet")

In [ ]:
df.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.494067  0.695173       7.690273        8.159422     7.8   
     2018-02-01  0.506174  0.726673       7.775459        8.036293     7.8   
     2018-03-01  0.493192  0.713110       7.890279        8.250671     7.8   
     2018-04-01  0.469532  0.713219       8.010917        8.123474     7.8   
     2018-05-01  0.483328  0.696213       8.044887        8.216409     7.7   

                  VU   CC   HA  fatalities  event_count  ...  \
iso3 month                                               ...   
AFG  2018-01-01  7.1  7.7  8.8      2822.0       1145.0  ...   
     2018-02-01  7.1  7.7  8.8      1829.0        920.0  ...   
     2018-03-01  7.1  7.7  8.8      2286.0        976.0  ...   
     2018-04-01  7.1  7.7  8.8      2794.0       1245.0  ...   
     2018-05-01  7.1  7.5  8.7      4264.0       1482.0  ...   

                Strategic developments  Violence against civilians  hdx_value  \
iso3 month                                                                      
AFG  2018-01-01                   37.0                        38.0        0.0   
     2018-02-01                   23.0                        26.0        1.0   
     2018-03-01                   14.0                        17.0        0.0   
     2018-04-01                   22.0                        22.0        0.0   
     2018-05-01                   18.0                        34.0        2.0   

                 hdx_alert_High concern  hdx_alert_Medium concern  \
iso3 month                                                          
AFG  2018-01-01                     0.0                       0.0   
     2018-02-01                     0.0                       1.0   
     2018-03-01                     0.0                       0.0   
     2018-04-01                     0.0                       0.0   
     2018-05-01                     1.0                       0.0   

                 monthly_displacement  rolling_3m_displacements  \
iso3 month                                                        
AFG  2018-01-01           7456.583577               7456.583577   
     2018-02-01          13918.956010              21375.539587   
     2018-03-01          15410.272726              36785.812313   
     2018-04-01          17198.881440              46528.110176   
     2018-05-01          39769.719735              72378.873901   

                 allocation-eligible  target_2m  start_conflict  
iso3 month                                                       
AFG  2018-01-01                    0        0.0               0  
     2018-02-01                    0        0.0               0  
     2018-03-01                    0        1.0               0  
     2018-04-01                    0        1.0               0  
     2018-05-01                    1        NaN               1  

[5 rows x 25 columns]

#### DERIVED HUMANITARIAN IMPACT

This section creates the features derived from the IDMC and ACLED datasets. Based on the literature review and the exploratory data analysis (EDA), we engineer features that capture recent humanitarian dynamics, including temporal trends, persistence, event intensity, conflict composition, and information extracted from the textual event descriptions. These features are intended to help the model identify the patterns that typically precede the onset of a new humanitarian crisis.

From the IDMC dataset, which contains the monthly number of new internal displacements, we generate several temporal features to capture both the recent level and the evolution of displacement. These include rolling averages, lagged values, changes between short- and long-term averages, binary threshold indicators, and persistence features that identify sustained periods of high displacement.

In [ ]:
# Basic feature engineering: rolling averages and lags for displacement
df['disp_6m_avg'] = df.groupby(level='iso3')['monthly_displacement'].transform(lambda x: x.rolling(6).mean())
df['disp_3m_avg'] = df.groupby(level='iso3')['monthly_displacement'].transform(lambda x: x.rolling(3).mean())
df['monthly_displacement_lag1'] = df.groupby('iso3')['monthly_displacement'].shift(1)
df['monthly_displacement_lag2'] = df.groupby('iso3')['monthly_displacement'].shift(2)

# Also features of the incremental change in displacement
df['disp_diff_3m_6m'] = df['disp_3m_avg'] - df['disp_6m_avg']
df['disp_change_lag1_to_now'] = df['monthly_displacement'] - df['monthly_displacement_lag1']
df['disp_change_lag2_to_now'] = df['monthly_displacement'] - df['monthly_displacement_lag2']

From the ACLED dataset, we perform three types of feature engineering. First, we generate temporal features from the `fatalities` variable, including rolling averages, lagged values, and trend indicators, following the same approach used for internal displacement. Second, we create binary and persistence features based on selected event types, as the EDA showed that different forms of violence have distinct relationships with humanitarian crises. Finally, we extract quantitative information from the `notes_acled` text field to capture evidence of forced displacement described in the event narratives.

In [ ]:
# Basic feature engineering: rolling averages and lags for fatalities
df['fat_6m_avg'] = df.groupby(level='iso3')['fatalities'].transform(lambda x: x.rolling(6).mean())
df['fat_3m_avg'] = df.groupby(level='iso3')['fatalities'].transform(lambda x: x.rolling(3).mean())
df['fatalities_lag1'] = df.groupby('iso3')['fatalities'].shift(1)
df['fatalities_lag2'] = df.groupby('iso3')['fatalities'].shift(2)

# Also features of the incremental change in fatalities
df['fat_diff_3m_6m'] = df['fat_3m_avg'] - df['fat_6m_avg']
df['fat_change_lag1_to_now'] = df['fatalities'] - df['fatalities_lag1']
df['fat_change_lag2_to_now'] = df['fatalities'] - df['fatalities_lag2']

In [ ]:
# Binary features for ACLED event types
df['bat_gr90'] = (df['Battles'] > 90).astype(int)
df['exp_gr150'] = (df['Explosions/Remote violence'] > 150).astype(int)

df['bat_gr90_and_exp_gr150'] = ((df['bat_gr90'] == 1) & (df['exp_gr150'] == 1)).astype(int)
df['bat_gr90_and_exp_gr150_last_5m'] = (
    df.groupby('iso3')['bat_gr90_and_exp_gr150']
    .transform(lambda x: x.rolling(window=5, min_periods=1).sum() > 0)
    .astype(int)
)

df["bat_civ"] = ((df["Battles"] > 0) & (df["Violence against civilians"] > 0)).astype(int)

# Index of Dangerousness of the conflict. We construct a monthly country-level conflict severity index using a weighted linear 
# combination of event frequencies multiplied by their respective danger coefficients. The weights will be proportioned to the 
# number of events of that time that tipically result in a allocation-eligible situation in the next two months.
weights = {
    'Battles': 5,
    'Explosions/Remote violence': 6,
    'Violence against civilians': 3,
    'Riots': 1,
    'Protests': 4,
    'Strategic developments': 2,  
}

event_type_columns = list(weights.keys())
weights_series = pd.Series(weights)
df['conflict_severity_index'] = df[event_type_columns].mul(weights_series).sum(axis=1)

In [ ]:
df.columns.tolist()

['risk_3',
 'risk_12',
 'logfat_risk_3',
 'logfat_risk_12',
 'INFORM',
 'VU',
 'CC',
 'HA',
 'fatalities',
 'event_count',
 'notes_acled',
 'Battles',
 'Explosions/Remote violence',
 'Protests',
 'Riots',
 'Strategic developments',
 'Violence against civilians',
 'hdx_value',
 'hdx_alert_High concern',
 'hdx_alert_Medium concern',
 'monthly_displacement',
 'rolling_3m_displacements',
 'allocation-eligible',
 'target_2m',
 'start_conflict',
 'disp_6m_avg',
 'disp_3m_avg',
 'monthly_displacement_lag1',
 'monthly_displacement_lag2',
 'disp_diff_3m_6m',
 'disp_change_lag1_to_now',
 'disp_change_lag2_to_now',
 'fat_6m_avg',
 'fat_3m_avg',
 'fatalities_lag1',
 'fatalities_lag2',
 'fat_diff_3m_6m',
 'fat_change_lag1_to_now',
 'fat_change_lag2_to_now',
 'bat_gr90',
 'exp_gr150',
 'bat_gr90_and_exp_gr150',
 'bat_gr90_and_exp_gr150_last_5m',
 'bat_civ',
 'conflict_severity_index']

To quantify information contained in the `notes_acled` column, we transform the event descriptions into numerical features using semantic text embeddings. To reduce the computational cost, we first split the raw text into individual phrases using the `" | "` separator and deduplicate them so that identical phrases are embedded only once. We then use the pre-trained Sentence Transformer `all-MiniLM-L6-v2` to generate vector representations and compute their cosine similarity with the target concept *"Civilian displacement caused by conflict or violence"*.

Each phrase whose similarity exceeds the selected threshold is considered to describe a displacement-related event. These phrase-level scores are subsequently aggregated at the monthly country level to produce features that summarize both the frequency and the intensity of displacement-related narratives in the ACLED event descriptions.

To reduce memory usage during this computationally intensive step, the similarities are computed in chunks of 100,000 phrases before the embedding model is explicitly cleared from memory. These numerical similarity scores are finally mapped back to the dataset into a new `displacement_score` column, mirroring the original text structure with the " | " delimiter. It is important to note that the underlying Python code for this step is kept separate, as we executed it directly in Google Colab to leverage GPU (CUDA) acceleration, which substantially reduces the execution time required to process hundreds of thousands of neural network embeddings.

In [ ]:
"""
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import drive
import gc
import torch

# Load data
drive.mount('/content/drive')
df = pd.read_parquet('/content/drive/MyDrive/Master_thesis/data_clean/complete_dataset.parquet')

# Load the model onto the Colab GPU
print("Loading model on CUDA...")
model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

# EXTRACT UNIQUE TEXTS
print("Extracting unique phrases...")
unique_texts_set = set()

for text_block in df["notes_acled"].dropna():
    phrases = str(text_block).split(" | ")
    for p in phrases:
        p_clean = p.strip()
        if p_clean != "" and p_clean != "nan":
            unique_texts_set.add(p_clean)

unique_texts = list(unique_texts_set)
print(f"Total unique phrases to process: {len(unique_texts)}")

# 
# CALCULATE SIMILARITIES IN CHUNKS 
print("Computing similarities in chunks...")

# Encode the TARGET FIRST AND ONLY ONCE
target_phrase = "Civilians fleeing violence, forced displacement, and population seeking refuge"
target_embedding = model.encode([target_phrase]) # Shape (1, 384)

similarities = []
chunk_size = 100000  # Process 100k phrases at a time to keep RAM near zero

for i in range(0, len(unique_texts), chunk_size):
    chunk = unique_texts[i:i+chunk_size]

    # Generate embeddings only for this mini-chunk
    chunk_emb = model.encode(chunk, batch_size=64, show_progress_bar=False)

    # Calculate similarity immediately and flatten
    chunk_sim = cosine_similarity(chunk_emb, target_embedding).flatten()

    # Save only the float numbers (incredibly lightweight)
    similarities.extend(chunk_sim)

    print(f"  Progress: {min(i + chunk_size, len(unique_texts))}/{len(unique_texts)} phrases processed.")

# Create the dictionary for super fast O(1) access
score_map = dict(zip(unique_texts, similarities))

# OPTIMIZATION: FREE MODEL FROM MEMORY BEFORE FINAL MAP
print("Freeing up GPU and model RAM to avoid crashes...")
del model
gc.collect()
torch.cuda.empty_cache()

#  MAP RESULTS BACK TO THE DATAFRAME
print("Mapping scores back to the dataset...")

def get_row_scores_as_string(text_block):
    if pd.isna(text_block) or str(text_block).strip() == "":
        return ""

    phrases = [p.strip() for p in str(text_block).split(" | ") if p.strip() != "" and p.strip() != "nan"]
    scores = [score_map.get(p, 0.0) for p in phrases]
    return ' | '.join(map(str, scores))

# Apply the function safely
df['displacement_score'] = df['notes_acled'].apply(get_row_scores_as_string)

# Save the results
print("Saving final file...")
df.to_parquet('/content/drive/MyDrive/Master_thesis/data_clean/final_scores.parquet')
print("Done! Successfully finished without burning the RAM!")
"""

'\nimport pandas as pd\nimport numpy as np\nfrom sentence_transformers import SentenceTransformer\nfrom sklearn.metrics.pairwise import cosine_similarity\nfrom google.colab import drive\nimport gc\nimport torch\n\n# Load data\ndrive.mount(\'/content/drive\')\ndf = pd.read_parquet(\'/content/drive/MyDrive/Master_thesis/data_clean/complete_dataset.parquet\')\n\n# Load the model onto the Colab GPU\nprint("Loading model on CUDA...")\nmodel = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")\n\n# EXTRACT UNIQUE TEXTS\nprint("Extracting unique phrases...")\nunique_texts_set = set()\n\nfor text_block in df["notes_acled"].dropna():\n    phrases = str(text_block).split(" | ")\n    for p in phrases:\n        p_clean = p.strip()\n        if p_clean != "" and p_clean != "nan":\n            unique_texts_set.add(p_clean)\n\nunique_texts = list(unique_texts_set)\nprint(f"Total unique phrases to process: {len(unique_texts)}")\n\n# \n# CALCULATE SIMILARITIES IN CHUNKS \nprint("Computing similariti

In [ ]:
# Load the data processed in the cloud
PATH_SCORES_CLOUD = "../data_clean/final_scores.parquet"
PATH_OUTPUT = "../data_clean/clima_acled_scores.parquet"

print("Loading cloud-processed data...")
df_cloud = pd.read_parquet(PATH_SCORES_CLOUD)

# Transfer the column with the score strings to your current dataframe
df['displacement_score'] = df_cloud['displacement_score']

del df_cloud 

CRITICAL_THRESHOLD = 0.53

# Function to calculate metrics from the score strings
print("Calculating aggregated metrics...")

def calculate_metrics_from_scores(score_string):
    if pd.isna(score_string) or str(score_string).strip() == "":
        return pd.Series([0.0, 0.0, 0.0, 0, 0])
        
    try:
        scores = [float(s.strip()) for s in str(score_string).split(" | ") if s.strip() != "" and s.strip() != "nan"]
    except ValueError:
        return pd.Series([0.0, 0.0, 0.0, 0, 0])
    
    if not scores:
        return pd.Series([0.0, 0.0, 0.0, 0, 0])
    
    score_max = max(scores)
    score_mean = sum(scores) / len(scores)
    score_sum = sum(scores) 
    disp_events = sum(1 for s in scores if s >= CRITICAL_THRESHOLD)
    total_events = len(scores)
    
    return pd.Series([score_max, score_mean, score_sum, disp_events, total_events])

# CREATE THE COLUMNS IN A NEW DATAFRAME CALLED df_final
print("Creating df_final with all integrated metrics...")

df_final = df.copy()

cols_to_create = [
    'acled_disp_score_max', 
    'acled_disp_score_mean', 
    'acled_semantic_intensity', 
    'acled_disp_events_count', 
    'acled_total_events'
]

# Apply the function to create new columns in df_final
df_final[cols_to_create] = df_final['displacement_score'].apply(calculate_metrics_from_scores)
df_final['acled_disp_events_ratio'] = df_final['acled_disp_events_count'] / df_final['acled_total_events'].replace(0, 1)
df_final = df_final.drop(columns=['displacement_score'])

print("Complete!")

Loading cloud-processed data...
Calculating aggregated metrics...
Creating df_final with all integrated metrics...
Complete!


In [ ]:
df_final.columns.tolist()

['risk_3',
 'risk_12',
 'logfat_risk_3',
 'logfat_risk_12',
 'INFORM',
 'VU',
 'CC',
 'HA',
 'fatalities',
 'event_count',
 'notes_acled',
 'Battles',
 'Explosions/Remote violence',
 'Protests',
 'Riots',
 'Strategic developments',
 'Violence against civilians',
 'hdx_value',
 'hdx_alert_High concern',
 'hdx_alert_Medium concern',
 'monthly_displacement',
 'rolling_3m_displacements',
 'allocation-eligible',
 'target_2m',
 'start_conflict',
 'disp_6m_avg',
 'disp_3m_avg',
 'monthly_displacement_lag1',
 'monthly_displacement_lag2',
 'disp_diff_3m_6m',
 'disp_change_lag1_to_now',
 'disp_change_lag2_to_now',
 'fat_6m_avg',
 'fat_3m_avg',
 'fatalities_lag1',
 'fatalities_lag2',
 'fat_diff_3m_6m',
 'fat_change_lag1_to_now',
 'fat_change_lag2_to_now',
 'bat_gr90',
 'exp_gr150',
 'bat_gr90_and_exp_gr150',
 'bat_gr90_and_exp_gr150_last_5m',
 'bat_civ',
 'conflict_severity_index',
 'acled_disp_score_max',
 'acled_disp_score_mean',
 'acled_semantic_intensity',
 'acled_disp_events_count',
 'acled_

Finally, we create a small set of features combining information from both IDMC and ACLED. Rather than relying only on the absolute numbers of fatalities, conflict events, and internal displacements, these features describe their relationship through normalized measures such as fatalities per event and displacements per fatality. This allows the model to capture the relative intensity and humanitarian impact of conflict events in addition to their absolute magnitude.

In [ ]:
df_final['lethality_rate'] = df_final['fatalities'].div(df_final['event_count']).replace([np.inf, -np.inf], 0).fillna(0)
df_final['disp_per_fatality'] = df_final['monthly_displacement'].div(df_final['fatalities']).replace([np.inf, -np.inf], 0).fillna(0)
df_final['disp_per_event'] = df_final['monthly_displacement'].div(df_final['event_count']).replace([np.inf, -np.inf], 0).fillna(0)

In [ ]:
df_final.columns.tolist()

['risk_3',
 'risk_12',
 'logfat_risk_3',
 'logfat_risk_12',
 'INFORM',
 'VU',
 'CC',
 'HA',
 'fatalities',
 'event_count',
 'notes_acled',
 'Battles',
 'Explosions/Remote violence',
 'Protests',
 'Riots',
 'Strategic developments',
 'Violence against civilians',
 'hdx_value',
 'hdx_alert_High concern',
 'hdx_alert_Medium concern',
 'monthly_displacement',
 'rolling_3m_displacements',
 'allocation-eligible',
 'target_2m',
 'start_conflict',
 'disp_6m_avg',
 'disp_3m_avg',
 'monthly_displacement_lag1',
 'monthly_displacement_lag2',
 'disp_diff_3m_6m',
 'disp_change_lag1_to_now',
 'disp_change_lag2_to_now',
 'fat_6m_avg',
 'fat_3m_avg',
 'fatalities_lag1',
 'fatalities_lag2',
 'fat_diff_3m_6m',
 'fat_change_lag1_to_now',
 'fat_change_lag2_to_now',
 'bat_gr90',
 'exp_gr150',
 'bat_gr90_and_exp_gr150',
 'bat_gr90_and_exp_gr150_last_5m',
 'bat_civ',
 'conflict_severity_index',
 'acled_disp_score_max',
 'acled_disp_score_mean',
 'acled_semantic_intensity',
 'acled_disp_events_count',
 'acled_

#### RISK ALARMS

This section engineers features from the EconAI and HDX Signals datasets, which provide early indicators of conflict risk and humanitarian deterioration. Rather than using only the original variables, we derive temporal features that capture recent trends, persistence, and abrupt changes in the underlying risk signals.

From EconAI, we use the variables `risk_3`, `risk_12`, `logfat_risk_3`, and `logfat_risk_12`. From HDX Signals, we use `hdx_alert_High concern`, `hdx_alert_Medium concern`, and `hdx_value`. 

Regarding EconAI, we follow the same feature engineering strategy used for the humanitarian indicators. We generate rolling averages and lagged values to capture the recent evolution of the different risk scores. In addition, motivated by the EDA, we create binary threshold indicators, persistence features that identify sustained periods of elevated risk, and variables measuring recent changes in the short-term risk estimates.

In [ ]:
df_final['risk3_6m_avg'] = df_final.groupby(level='iso3')['risk_3'].transform(lambda x: x.rolling(6).mean())
df_final['risk3_3m_avg'] = df_final.groupby(level='iso3')['risk_3'].transform(lambda x: x.rolling(3).mean())
df_final['risk3_lag1'] = df_final.groupby('iso3')['risk_3'].shift(1)
df_final['risk3_lag2'] = df_final.groupby('iso3')['risk_3'].shift(2)

df_final['risk12_6m_avg'] = df_final.groupby(level='iso3')['risk_12'].transform(lambda x: x.rolling(6).mean())
df_final['risk12_3m_avg'] = df_final.groupby(level='iso3')['risk_12'].transform(lambda x: x.rolling(3).mean())
df_final['risk12_lag1'] = df_final.groupby('iso3')['risk_12'].shift(1)
df_final['risk12_lag2'] = df_final.groupby('iso3')['risk_12'].shift(2)

df_final['logfat_risk3_6m_avg'] = df_final.groupby(level='iso3')['logfat_risk_3'].transform(lambda x: x.rolling(6).mean())
df_final['logfat_risk3_3m_avg'] = df_final.groupby(level='iso3')['logfat_risk_3'].transform(lambda x: x.rolling(3).mean())
df_final['logfat_risk3_lag1'] = df_final.groupby('iso3')['logfat_risk_3'].shift(1)
df_final['logfat_risk3_lag2'] = df_final.groupby('iso3')['logfat_risk_3'].shift(2)

df_final['logfat_risk12_6m_avg'] = df_final.groupby(level='iso3')['logfat_risk_12'].transform(lambda x: x.rolling(6).mean())
df_final['logfat_risk12_3m_avg'] = df_final.groupby(level='iso3')['logfat_risk_12'].transform(lambda x: x.rolling(3).mean())
df_final['logfat_risk12_lag1'] = df_final.groupby('iso3')['logfat_risk_12'].shift(1)
df_final['logfat_risk12_lag2'] = df_final.groupby('iso3')['logfat_risk_12'].shift(2) 


Based on the patterns identified during the EDA, we also engineer several indicator variables that identify periods of elevated conflict risk and sustained fatality forecasts. Finally, we include short-term changes in the risk scores to capture sudden increases in predicted conflict intensity.

In [ ]:
df_final['risk_gt_025'] = (df_final['risk_3'] > 0.25).astype(int)
df_final['logfat_risk_3_gt_4'] = (df_final['logfat_risk_3'] > 4).astype(int)

# Log fat risk 3 above 4 for the last 5 months (including the current month)
df_final['logfat_risk_3_high_last_5m'] = (
    df_final.groupby('iso3')['logfat_risk_3_gt_4']
    .transform(lambda x: x.rolling(window=5, min_periods=1).sum() > 0)
    .astype(int)
)

# Log fat risk 12 above 5 for the last 5 months (including the current month)
df_final['logfat_risk_12_above_5'] = (df_final['logfat_risk_12'] > 5).astype(int)
df_final['logfat_risk_12_high_last_5m'] = (
    df_final.groupby('iso3')['logfat_risk_12_above_5']
    .transform(lambda x: x.rolling(window=5, min_periods=1).sum() > 0)
    .astype(int)
)

# Change in risk the last 2 months
df_final['risk3_change_2m'] = df_final.groupby('iso3')['risk_3'].diff(2)
df_final['risk12_change_2m'] = df_final.groupby('iso3')['risk_12'].diff(2)

# Change in logfat risk the last 2 months
df_final['logfat_risk3_change_2m'] = df_final.groupby('iso3')['logfat_risk_3'].diff(2)
df_final['logfat_risk12_change_2m'] = df_final.groupby('iso3')['logfat_risk_12'].diff(2)


And then, from the For the HDX Signals dataset, we derive features summarizing both the frequency and the intensity of humanitarian alerts. Specifically, we compute the total number of medium- and high-concern alerts, rolling three-month counts for each alert category, and the average alert value per issued alert. These variables capture the persistence and severity of humanitarian warnings over recent months. Signals:


In [ ]:
df_final['hdx_med_high_count'] = df_final["hdx_alert_High concern"] + df_final["hdx_alert_Medium concern"]
df_final['hdx_3m_sum'] = df_final.groupby(level='iso3')['hdx_med_high_count'].transform(lambda x: x.rolling(3).sum())
df_final['hdx_medium_3m_sum'] = df_final.groupby(level='iso3')['hdx_alert_Medium concern'].transform(lambda x: x.rolling(3).sum())
df_final['hdx_high_3m_sum'] = df_final.groupby(level='iso3')['hdx_alert_High concern'].transform(lambda x: x.rolling(3).sum())

df_final['mean_value_hdx'] = df_final['hdx_value'] / df_final['hdx_med_high_count']
df_final['mean_value_hdx'] = df_final['mean_value_hdx'].replace([np.inf, -np.inf], np.nan).fillna(0)

In [ ]:
df_final.columns.tolist()

['risk_3',
 'risk_12',
 'logfat_risk_3',
 'logfat_risk_12',
 'INFORM',
 'VU',
 'CC',
 'HA',
 'fatalities',
 'event_count',
 'notes_acled',
 'Battles',
 'Explosions/Remote violence',
 'Protests',
 'Riots',
 'Strategic developments',
 'Violence against civilians',
 'hdx_value',
 'hdx_alert_High concern',
 'hdx_alert_Medium concern',
 'monthly_displacement',
 'rolling_3m_displacements',
 'allocation-eligible',
 'target_2m',
 'start_conflict',
 'disp_6m_avg',
 'disp_3m_avg',
 'monthly_displacement_lag1',
 'monthly_displacement_lag2',
 'disp_diff_3m_6m',
 'disp_change_lag1_to_now',
 'disp_change_lag2_to_now',
 'fat_6m_avg',
 'fat_3m_avg',
 'fatalities_lag1',
 'fatalities_lag2',
 'fat_diff_3m_6m',
 'fat_change_lag1_to_now',
 'fat_change_lag2_to_now',
 'bat_gr90',
 'exp_gr150',
 'bat_gr90_and_exp_gr150',
 'bat_gr90_and_exp_gr150_last_5m',
 'bat_civ',
 'conflict_severity_index',
 'acled_disp_score_max',
 'acled_disp_score_mean',
 'acled_semantic_intensity',
 'acled_disp_events_count',
 'acled_

#### INFORM

Based on the exploratory data analysis (EDA), we create a binary indicator identifying months in which the INFORM index exceeds 0.6, together with a persistence feature indicating whether this threshold has been exceeded at least once during the previous five months (including the current month). These features capture both periods of elevated structural risk and the persistence of high-risk conditions over time.

In [ ]:
# INFORM Index above 0.6 for the last 5 months (including the current month)
df_final['INFORM_above_06'] = (df_final['INFORM'] > 0.6).astype(int)
df_final['INFORM_high_last_5m'] = (
    df_final.groupby('iso3')['INFORM_above_06']
    .transform(lambda x: x.rolling(window=5, min_periods=1).sum() > 0)
    .astype(int)
)

In [ ]:
df_final.columns.tolist()

['risk_3',
 'risk_12',
 'logfat_risk_3',
 'logfat_risk_12',
 'INFORM',
 'VU',
 'CC',
 'HA',
 'fatalities',
 'event_count',
 'notes_acled',
 'Battles',
 'Explosions/Remote violence',
 'Protests',
 'Riots',
 'Strategic developments',
 'Violence against civilians',
 'hdx_value',
 'hdx_alert_High concern',
 'hdx_alert_Medium concern',
 'monthly_displacement',
 'rolling_3m_displacements',
 'allocation-eligible',
 'target_2m',
 'start_conflict',
 'disp_6m_avg',
 'disp_3m_avg',
 'monthly_displacement_lag1',
 'monthly_displacement_lag2',
 'disp_diff_3m_6m',
 'disp_change_lag1_to_now',
 'disp_change_lag2_to_now',
 'fat_6m_avg',
 'fat_3m_avg',
 'fatalities_lag1',
 'fatalities_lag2',
 'fat_diff_3m_6m',
 'fat_change_lag1_to_now',
 'fat_change_lag2_to_now',
 'bat_gr90',
 'exp_gr150',
 'bat_gr90_and_exp_gr150',
 'bat_gr90_and_exp_gr150_last_5m',
 'bat_civ',
 'conflict_severity_index',
 'acled_disp_score_max',
 'acled_disp_score_mean',
 'acled_semantic_intensity',
 'acled_disp_events_count',
 'acled_

#### CERF SIGNALS

Based on the CERF operational guidance, countries can experience humanitarian crises under two broad contexts: **protracted crises** and **hard-onset crises**. Since these situations typically exhibit different early warning patterns, we first classify each country-month into one of these two states before constructing the corresponding signal features.

To identify protracted crises, we follow the criterion that a country is considered to be in a protracted state when the EconAI 3-month risk score remains above 0.25 for 12 consecutive months. Country-months that do not satisfy this condition are classified as **hard onset**.

We then derive early warning signals tailored to each state.

**Protracted crisis signals**

- The 3-month EconAI risk score increases by at least **0.05** over the previous three months while remaining above **0.30**.
- Monthly internal displacement or fatalities exceed **1.5 times** their rolling six-month average for **two consecutive months**.
- At least **three** medium- or high-concern HDX alerts are issued within a three-month period.

**Hard-onset crisis signals**

- The 3-month EconAI risk score increases by at least **0.15** within two months.
- Monthly internal displacement exceeds **three times** its rolling six-month average. When displacement data are unavailable, the same criterion is applied to fatalities instead.

These engineered features are intended to capture the distinct warning patterns associated with each crisis type while remaining consistent with the operational logic used by CERF.

In [ ]:
# 1. CLASSIFY THE STATE (Protracted vs Hard Onset)

# We use rolling sum of 12 on the boolean column. If sum is 12, it was True for 12 consecutive months.
df_final['is_protracted'] = df_final.groupby(level='iso3')['risk_gt_025'].transform(
    lambda x: x.rolling(window=12, min_periods=12).sum() == 12
).astype(int)

df_final['state'] = np.where(df_final['is_protracted']==1, 'Protracted', 'Hard onset')

df_final["is_protracted"].value_counts()

is_protracted
0    16508
1     1092
Name: count, dtype: int64

In [ ]:
columna_target = 'target_2m' 

# Keep only positive targets for analysis
positives = df_final[df_final[columna_target] == 1]
counts = positives['is_protracted'].value_counts()

print("Of the real crises (Target = 1):")
print(f"Hard Onset (is_protracted = 0): {counts.get(0, 0)} cases")
print(f"Protracted (is_protracted = 1): {counts.get(1, 0)} cases")
print("-" * 40)

Of the real crises (Target = 1):
Hard Onset (is_protracted = 0): 120 cases
Protracted (is_protracted = 1): 92 cases
----------------------------------------


In [ ]:
# 3. PROTRACTED SIGNALS EVALUATION

# P1: Risk increases by >= 0.05 over 3 months while already > 0.3
df_final['p_sig1'] = ((df_final.groupby(level='iso3')['risk_3'].diff(3) >= 0.05) & (df_final['risk_3'] > 0.3)).astype(int)

# P2: Displacement or fatalities >= 1.5x their 6-month average for 2 consecutive months
spike_1_5x = (df_final['monthly_displacement'] >= 1.5 * df_final['disp_6m_avg']) | (df_final['fatalities'] >= 1.5 * df_final['fat_6m_avg'])

# Agrupamos la Serie directamente usando su propio índice (level='iso3')
df_final['p_sig2'] = spike_1_5x.groupby(level='iso3').transform(
    lambda x: x.rolling(2).sum() == 2
).astype(int)
# P3: >= 3 Medium or High HDX signals within 90 days
df_final['p_sig3'] = (df_final['hdx_3m_sum'] >= 3).astype(int)

# Combine Protracted signals (True if any is True)
df_final['protracted_signal'] = (df_final['p_sig1'] | df_final['p_sig2'] | df_final['p_sig3']).astype(int)


In [ ]:
# 4. HARD ONSET SIGNALS EVALUATION

# H1: Risk increases by >= 0.15 within 2 months
df_final['h_sig1'] = (df_final.groupby(level='iso3')['risk_3'].diff(2) >= 0.15).astype(int)

# H2: Displacement > 3x 6-month avg. If missing, use fatalities > 3x 6-month avg.
disp_spike_3x = df_final['monthly_displacement'] > 3 * df_final['disp_6m_avg']
fat_spike_3x = df_final['fatalities'] > 3 * df_final['fat_6m_avg']

# np.where lets us use the fatality logic ONLY when displacement is NaN
df_final['h_sig2'] = np.where(df_final['monthly_displacement'].isna(), fat_spike_3x, disp_spike_3x).astype(int)

# Combine Hard Onset signals
df_final['hard_onset_signal'] = (df_final['h_sig1'] | df_final['h_sig2']).astype(int)


In [ ]:
df_final.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.494067  0.695173       7.690273        8.159422     7.8   
     2018-02-01  0.506174  0.726673       7.775459        8.036293     7.8   
     2018-03-01  0.493192  0.713110       7.890279        8.250671     7.8   
     2018-04-01  0.469532  0.713219       8.010917        8.123474     7.8   
     2018-05-01  0.483328  0.696213       8.044887        8.216409     7.7   

                  VU   CC   HA  fatalities  event_count  ...  \
iso3 month                                               ...   
AFG  2018-01-01  7.1  7.7  8.8      2822.0       1145.0  ...   
     2018-02-01  7.1  7.7  8.8      1829.0        920.0  ...   
     2018-03-01  7.1  7.7  8.8      2286.0        976.0  ...   
     2018-04-01  7.1  7.7  8.8      2794.0       1245.0  ...   
     2018-05-01  7.1  7.5  8.7      4264.0       1482.0  ...   

                INFORM_high_last_5m  is_protracted       state  p_sig1  \
iso3 month                                                               
AFG  2018-01-01                   1              0  Hard onset       0   
     2018-02-01                   1              0  Hard onset       0   
     2018-03-01                   1              0  Hard onset       0   
     2018-04-01                   1              0  Hard onset       0   
     2018-05-01                   1              0  Hard onset       0   

                 p_sig2  p_sig3  protracted_signal  h_sig1  h_sig2  \
iso3 month                                                           
AFG  2018-01-01       0       0                  0       0       0   
     2018-02-01       0       0                  0       0       0   
     2018-03-01       0       0                  0       0       0   
     2018-04-01       0       0                  0       0       0   
     2018-05-01       0       0                  0       0       0   

                 hard_onset_signal  
iso3 month                          
AFG  2018-01-01                  0  
     2018-02-01                  0  
     2018-03-01                  0  
     2018-04-01                  0  
     2018-05-01                  0  

[5 rows x 95 columns]

In [ ]:
display(pd.DataFrame(df_final.columns, columns=['Column Name']))

,Column Name
0,risk_3
1,risk_12
2,logfat_risk_3
3,logfat_risk_12
4,INFORM
...,...
90,p_sig3
91,protracted_signal
92,h_sig1
93,h_sig2


In [ ]:
df_final.columns.tolist()

['risk_3',
 'risk_12',
 'logfat_risk_3',
 'logfat_risk_12',
 'INFORM',
 'VU',
 'CC',
 'HA',
 'fatalities',
 'event_count',
 'notes_acled',
 'Battles',
 'Explosions/Remote violence',
 'Protests',
 'Riots',
 'Strategic developments',
 'Violence against civilians',
 'hdx_value',
 'hdx_alert_High concern',
 'hdx_alert_Medium concern',
 'monthly_displacement',
 'rolling_3m_displacements',
 'allocation-eligible',
 'target_2m',
 'start_conflict',
 'disp_6m_avg',
 'disp_3m_avg',
 'monthly_displacement_lag1',
 'monthly_displacement_lag2',
 'disp_diff_3m_6m',
 'disp_change_lag1_to_now',
 'disp_change_lag2_to_now',
 'fat_6m_avg',
 'fat_3m_avg',
 'fatalities_lag1',
 'fatalities_lag2',
 'fat_diff_3m_6m',
 'fat_change_lag1_to_now',
 'fat_change_lag2_to_now',
 'bat_gr90',
 'exp_gr150',
 'bat_gr90_and_exp_gr150',
 'bat_gr90_and_exp_gr150_last_5m',
 'bat_civ',
 'conflict_severity_index',
 'acled_disp_score_max',
 'acled_disp_score_mean',
 'acled_semantic_intensity',
 'acled_disp_events_count',
 'acled_

In [ ]:
df_final.to_parquet("../data_clean/final_data.parquet")

### CHECKS ON THE FINAL DATASET

Before exporting the engineered dataset, we perform a final validation to ensure that the feature engineering pipeline has produced a complete and internally consistent dataset. Specifically, we verify the structure of the final dataframe, inspect the generated feature set, and confirm that the temporal coverage is complete and free of unexpected gaps.

In [ ]:
display(pd.DataFrame(df_final.columns, columns=['Column Name']))

,Column Name
0,risk_3
1,risk_12
2,logfat_risk_3
3,logfat_risk_12
4,INFORM
...,...
90,p_sig3
91,protracted_signal
92,h_sig1
93,h_sig2


In [ ]:
print(df_final.columns.tolist())

['risk_3', 'risk_12', 'logfat_risk_3', 'logfat_risk_12', 'INFORM', 'VU', 'CC', 'HA', 'fatalities', 'event_count', 'notes_acled', 'Battles', 'Explosions/Remote violence', 'Protests', 'Riots', 'Strategic developments', 'Violence against civilians', 'hdx_value', 'hdx_alert_High concern', 'hdx_alert_Medium concern', 'monthly_displacement', 'rolling_3m_displacements', 'allocation-eligible', 'target_2m', 'start_conflict', 'disp_6m_avg', 'disp_3m_avg', 'monthly_displacement_lag1', 'monthly_displacement_lag2', 'disp_diff_3m_6m', 'disp_change_lag1_to_now', 'disp_change_lag2_to_now', 'fat_6m_avg', 'fat_3m_avg', 'fatalities_lag1', 'fatalities_lag2', 'fat_diff_3m_6m', 'fat_change_lag1_to_now', 'fat_change_lag2_to_now', 'bat_gr90', 'exp_gr150', 'bat_gr90_and_exp_gr150', 'bat_gr90_and_exp_gr150_last_5m', 'bat_civ', 'conflict_severity_index', 'acled_disp_score_max', 'acled_disp_score_mean', 'acled_semantic_intensity', 'acled_disp_events_count', 'acled_total_events', 'acled_disp_events_ratio', 'lethal

The following checks verify that the final dataset contains the expected geographical and temporal coverage. We confirm the total number of countries and monthly observations included in the analysis and ensure that the global timeline contains no missing months between the first and last observation.

In [ ]:
df_check = df_final.reset_index() if isinstance(df_final.index, pd.MultiIndex) else df_final.copy()
df_check['month'] = pd.to_datetime(df_check['month'])

# Verify the geographical and temporal coverage of the final dataset
num_countries = df_check['iso3'].nunique()
num_months = df_check['month'].nunique()

print("=" * 50)
print(f"Dataset statistics:")
print(f"   -> Number of unique countries (iso3): {num_countries}")
print(f"   -> Total number of unique months:  {num_months}")
print("=" * 50)

# Ensure that the global monthly timeline is complete
min_date = df_check['month'].min()
max_date = df_check['month'].max()
expected_range = pd.date_range(start=min_date, end=max_date, freq='MS') # 'MS' = Month Start
missing_months = [m.strftime('%Y-%m-%d') for m in expected_range if m not in df_check['month'].values]

print(f"Temporal analysis (From {min_date.strftime('%Y-%m')} to {max_date.strftime('%Y-%m')}):")
print(f"   -> Expected months: {len(expected_range)}")

if len(missing_months) == 0:
    print("   -> PERFECT: No temporal gaps found in the global dataset. All months are covered.")
else:
    print(f"   -> ALERT: {len(missing_months)} months are missing in the global dataset!")
    print(f"   -> Missing months: {missing_months}")
print("=" * 50)

Dataset statistics:
   -> Number of unique countries (iso3): 176
   -> Total number of unique months:  100
Temporal analysis (From 2018-01 to 2026-04):
   -> Expected months: 100
   -> PERFECT: No temporal gaps found in the global dataset. All months are covered.


### INTERACTIVE PLOT 

The following interactive visualization summarizes the complete timeline for a selected country. It combines the observed monthly internal displacements, the periods in which the country is considered allocation-eligible according to the CERF criteria, the months labeled as early-warning observations (one to two months before the start of a new eligibility episode), and the actual CERF Rapid Response allocations.

Use the dropdown menu to select a country. Hovering over the plot displays the underlying values for each observation. This visualization provides a qualitative validation of the target construction and feature engineering process, making it possible to inspect whether the generated warning periods precede allocation-eligible episodes and how they relate to the timing of real CERF allocations.

In [23]:
df = pd.read_parquet("../data_clean/final_data.parquet")
df = df.reset_index()
cerf = pd.read_csv("../data_clean/cerf_clean.csv")
cerf['Allocation Date'] = pd.to_datetime(cerf['Allocation Date'], format='mixed')

In [24]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output

# Define the function to create the country timeline figure
def create_country_timeline_fig(country):
    df_plot = df.copy()
    if 'iso3' not in df_plot.columns and df_plot.index.names and 'iso3' in df_plot.index.names:
        df_plot = df_plot.reset_index()
        
    df_plot = df_plot[df_plot['iso3'] == country].copy()
    df_plot['month'] = pd.to_datetime(df_plot['month'])
    df_plot = df_plot.sort_values('month')
    
    if df_plot.empty:
        print(f"No data available for country: {country}")
        return None

    valid_disp = df_plot.dropna(subset=['monthly_displacement'])
    if not valid_disp.empty:
        min_date = valid_disp['month'].min() - pd.Timedelta(days=30)
        max_date = valid_disp['month'].max() + pd.Timedelta(days=30)
    else:
        min_date = df_plot['month'].min()
        max_date = df_plot['month'].max()

    df_plot['disp_log1p'] = np.log1p(df_plot['monthly_displacement'].fillna(0))
    
    max_y = df_plot['disp_log1p'].max()
    top_y_pos = max_y * 1.1 if max_y > 0 else 1.0

    if 'target_2m' in df_plot.columns:
        warning_months = df_plot[df_plot['target_2m'] == 1]['month']
        warning_dates = warning_months + pd.Timedelta(days=0)
    else:
        warning_dates = pd.Series(dtype='datetime64[ns]')

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df_plot['month'],
        y=df_plot['disp_log1p'],
        customdata=df_plot['monthly_displacement'], 
        mode='lines', 
        name='Displacements [log(1+x)]',
        line=dict(color='purple', width=2.5),
        yaxis='y1',
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Displacements (Real):</b> %{customdata:,.0f}<br><b>log(1+x):</b> %{y:.2f}<extra></extra>'
    ))

    if 'allocation-eligible' in df_plot.columns:
        conflict_months = df_plot[df_plot['allocation-eligible'] == 1]['month']
        for c_month in conflict_months:
            fig.add_vrect(
                x0=c_month - pd.Timedelta(days=15), 
                x1=c_month + pd.Timedelta(days=15),
                fillcolor="salmon",
                opacity=0.2,
                layer="below",
                line_width=0,
            )
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Allocation-eligible Zone',
            line=dict(color='salmon', width=10), opacity=0.3, yaxis='y1'
        ))

    if not warning_dates.empty:
        for w_date in warning_dates:
            fig.add_shape(
                type="line", x0=w_date, x1=w_date, y0=0, y1=1,
                xref="x", yref="paper", line=dict(width=2.5, dash="dash", color="red"),
            )
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Warning (1-2m Pre-Conflict)',
            line=dict(color='red', width=2.5, dash='dash'), yaxis='y1'
        ))

    cerf_country = cerf[cerf['iso3'] == country].copy()
    if not cerf_country.empty:
        cerf_country['plot_date'] = cerf_country['Allocation Date'].dt.to_period('M').dt.to_timestamp() + pd.Timedelta(days=14)
        
        fig.add_trace(go.Scatter(
            x=cerf_country['plot_date'],
            y=[top_y_pos] * len(cerf_country),
            mode='markers',
            marker=dict(symbol='circle', size=12, color='gold', line=dict(width=1.5, color='darkorange')),
            name='CERF Allocation',
            hovertemplate='<b>CERF Date:</b> %{x|%d %b %Y}<br><b>Amount:</b> $%{customdata:,.0f}<extra></extra>',
            customdata=cerf_country['Amount Approved']
        ))

    fig.update_layout(
        title=dict(text=f'<b>Full Crisis Timeline Overview: {country}</b>', font=dict(size=22), x=0.05),
        margin=dict(l=60, r=100, t=110, b=60), 
        height=600,
        plot_bgcolor='white',
        hovermode="closest",
        xaxis=dict(
            title=dict(text="<b>Date</b>"), showgrid=False, dtick="M3",
            tickformat="%b %Y", tickangle=45, range=[min_date, max_date]
        ),
        yaxis=dict(
            title=dict(text="<b>log(1 + Displacements)</b>", font=dict(color="purple")),
            tickfont=dict(color="purple"), showgrid=True, gridcolor='lightgrey',
            rangemode="tozero", range=[0, top_y_pos * 1.05] 
        ),
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5,
            bgcolor='rgba(255,255,255,0.8)', bordercolor='lightgrey', borderwidth=1
        )
    )

    return fig

# Extract unique countries for the dropdown menu
if 'iso3' in df.columns:
    countries = sorted(df['iso3'].dropna().unique().tolist())
else:
    countries = sorted(df.index.get_level_values('iso3').dropna().unique().tolist())

# Configure the dropdown widget and output area
dropdown = widgets.Dropdown(options=countries, value='AFG' if 'AFG' in countries else countries[0], description='Country:')
out = widgets.Output()

def on_dropdown_change(change):
    with out:
        clear_output(wait=True) 
        fig = create_country_timeline_fig(change['new'])
        if fig:
            display(fig)

dropdown.observe(on_dropdown_change, names='value')

# Show the dropdown and the initial figure
display(dropdown, out)

with out:
    fig = create_country_timeline_fig(dropdown.value)
    if fig:
        display(fig)

Dropdown(description='Country:', options=('AFG', 'AGO', 'ALB', 'ARE', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE', 'BDI'…

Output()